# <center><font color=purple>COMP4703 Natural Language Processing</font></center>


# <center><font color=purple>Assignment 1 - Part of Speech Tagging and Named Entity Recognition (15 Marks)</font></center>


---


---


In this assignment, you will implement Part-of-Speech and Named Entity parsers using a popular NLP
library - [**spaCy**](https://spacy.io/). The API is well documented and it should be relatively straightforward
for you to get both parsers to work. The one important caveat is that you must be careful when
you tokenize the input text. The input text is already pre-parsed, and since tokenizers change regularly and make different decisions
when parsing text, you **must** ensure that you enable "white space" only parsing options, otherwise your output file will
not be properly aligned with the evaluation labels, and the evaluation function will fail.

For debugging and testing purposes, I have included a `test_local.txt` and the gold labels `test_local.tsv`. which
include tokens that will be parsed incorrectly without a "white space" only parser enabled. I would stronly urge you to do your initial development using these two files before you do your final run using ``test.tsv''. I have also included the code to dump the two sample files line by line below so that you will understand the format. Additional information about the input format is inlined below.

In the cells below, I have included the headers you will need, as well as the code to download models automatically --
assuming that you have installed the libraries listed in the README.md file included in the zip file, which explains how to
create an anaconda virtual environment that will contain everything you need to complete the assignment. You should not need
to change any of the libraries or headers. In addition, there is a large block of convenience functions that I have inlcuded
to enable the creation of a log file, write output files, and convert label formats. In the template to execute the functions
below. If there is a section marked DO NOT MODIFY! then you should not make any changes to that block, or you will
lose marks. To complete the exercise, you should only need to write functions that are clearly marked as "YOUR CODE GOES HERE".


---


### <font color=purple>IMPORT HEADERS. DO NOT MODIFY!</color>


---


In [ ]:
import sys, os, gzip
from tqdm import tqdm
import time
import logging
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import f1_score, recall_score, precision_score

# import matplotlib.pyplot as plt
# from sklearn.metrics import RocCurveDisplay
import numpy

import spacy
import spacy_transformers
from spacy.tokens import Doc

import re
from spacy.tokenizer import Tokenizer


import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

splogger = logging.getLogger("spacy")
splogger.setLevel(logging.ERROR)

numpy.set_printoptions(threshold=sys.maxsize)

# from .autonotebook import tqdm as notebook_tqdm

---


### <font color=purple>GLOBAL LOGFILE INITIALISATION. DO NOT MODIFY!<color>


---


In [ ]:
# Move logfile to backup each time the kernel is reran
backup_logfile = "COMP4703A1.log.bak"
logfile = "COMP4703A1.log"
if os.path.exists(backup_logfile):
    os.remove(backup_logfile)
if os.path.exists(logfile):
    os.rename(logfile, backup_logfile)

# Initialise logger
logger = logging.getLogger(__name__)
logging.basicConfig(filename="COMP4703A1.log", level=logging.DEBUG)
# Example logging output types you can use.
# logger.debug('This debug message should go to the log file')
# logger.info('Info should be this')
# logger.warning('And this warning, too')
# logger.error('And an error')
# END GLOBAL LOGFILE INITIALISATION

---


### <font color=purple>BEGIN UTILITY FUNCTIONS. DO NOT MODIFY!</font>


---


In [ ]:
# Decorator function to compute function running time. Just add @timer just before a function and
# it will automatically compute and print the runtime.
def nlptimer(func):
    """Function decorator to measure execution time of functions."""

    def wrapper(*args, **kwargs):
        start_time = time.time()  # Start time
        result = func(*args, **kwargs)  # Execute the function
        end_time = time.time()  # End time
        execution_time = end_time - start_time  # Calculate execution time
        print(f"\n{func.__name__} executed in: {execution_time} seconds\n")
        logger.info(f"\n{func.__name__} executed in: {execution_time} seconds\n")
        return result

    return wrapper


# Try to fix the default spaCy NER tags to align with the labelled data.
@nlptimer
def fix_default_ner_tags(output_file):
    ontonotes_to_conll = {
        "GPE": "LOCATION",
        "LOC": "LOCATION",
        "ORG": "ORGANIZATION",
        "NORP": "MISC",
        "FAC": "MISC",
        "PRODUCT": "MISC",
        "EVENT": "MISC",
        "WORK_OF_ART": "MISC",
        "LAW": "MISC",
        "LANGUAGE": "MISC",
        # The following labels do not exist in CoNLL-2003 and are thus ignored
        "QUANTITY": "O",
        "CARDINAL": "O",
    }
    rv = []
    for ln in tqdm(output_file.split("\n")):
        ln = ln.strip()
        if ln:
            fld = ln.split("\t")
            i = fld[0]
            w = fld[1]
            n = fld[2]
            if n in ontonotes_to_conll:
                n = ontonotes_to_conll[n]
            rv.append("{}\t{}\t{}".format(i, w, n))
        else:
            rv.append("")
    return "\n".join(rv)


# Evaluate a POS run
@nlptimer
def evaluate_pos_run(output_file):
    label_names = [
        "-LRB-",
        "-RRB-",
        ",",
        ":",
        ".",
        "''",
        "#",
        "``",
        "$",
        "CC",
        "CD",
        "DT",
        "EX",
        "FW",
        "IN",
        "JJ",
        "JJR",
        "JJS",
        "LS",
        "MD",
        "NN",
        "NNP",
        "NNPS",
        "NNS",
        "PDT",
        "POS",
        "PRP",
        "PRP$",
        "RB",
        "RBR",
        "RBS",
        "RP",
        "SYM",
        "TO",
        "UH",
        "VB",
        "VBD",
        "VBG",
        "VBN",
        "VBP",
        "VBZ",
        "WDT",
        "WP",
        "WP$",
        "WRB",
    ]

    gold_file = "test-local.tsv.gz"
    y_pred_pos = []
    y_true_pos = []

    print("[INFO] Reading POS data for evaluation.")
    logger.info("[INFO] Reading POS data for evaluation.")
    with gzip.open(gold_file, "rt") as fh:
        for i, ln in enumerate(fh):
            if i == 0:
                if len(ln.strip().split("\t")) != 4:
                    print(
                        "[ERROR]: The file: {} is not formatted correctly.".format(
                            gold_file
                        )
                    )
                    logger.error(
                        "[ERROR]: The file: {} is not formatted correctly.".format(
                            gold_file
                        )
                    )
                    sys.exit(-1)
            ln = ln.strip()
            if ln:
                fld = ln.split("\t")
                y_true_pos.append(fld[2])

    ref_len = i

    for i, ln in enumerate(output_file.split("\n")):
        if i == 0:
            if len(ln.strip().split("\t")) != 3:
                print(
                    "[ERROR]: The data: {} is not formatted correctly.".format(
                        "output_file"
                    )
                )
                logger.error(
                    "[ERROR]: The data: {} is not formatted correctly.".format(
                        "output_file"
                    )
                )
                sys.exit(-1)
        ln = ln.strip()
        if ln:
            fld = ln.split("\t")
            y_pred_pos.append(fld[2])

    pred_len = i

    if pred_len != ref_len:
        print(
            "[ERROR]: The reference file and prediction file are not the same length."
        )
        logger.error(
            "[ERROR]: The reference file and prediction file are not the same length."
        )
        sys.exit(-1)

    print("[INFOO] Classification report for POS predictions.")
    print(
        classification_report(
            y_true_pos, y_pred_pos, labels=label_names, zero_division=1
        )
    )
    logger.info("[INFO] Classification report for POS predictions.")
    logger.info(
        classification_report(
            y_true_pos, y_pred_pos, labels=label_names, zero_division=1
        )
    )

    df = confusion_matrix(y_true_pos, y_pred_pos, labels=label_names)
    print("[INFO} Confusion Matrix for POS Predictions")
    print(df)
    logger.info("[INFO] Confusion Matrix for POS Predictions")
    logger.info(df)

    label_binarizer = LabelBinarizer().fit(y_true_pos)
    y_onehot_test = label_binarizer.transform(y_pred_pos)
    x_onehot_true = label_binarizer.transform(y_true_pos)
    print("[INFO] POS Weighted Micro Scores")
    logger.info("[INFO] POS Weighted Micro Scores")
    micro_roc_auc_ovr = roc_auc_score(
        y_onehot_test, x_onehot_true, multi_class="ovr", average="micro"
    )
    print(
        "POS Recall = {}".format(
            round(
                recall_score(
                    y_true_pos, y_pred_pos, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    print(
        "POS Precision = {}".format(
            round(
                precision_score(
                    y_true_pos, y_pred_pos, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    print(
        "POS F1 = {}".format(
            round(
                f1_score(y_true_pos, y_pred_pos, average="weighted", zero_division=1), 3
            )
        )
    )
    print("POS AUC = {}".format(round(micro_roc_auc_ovr, 3)))
    logger.info(
        "POS Recall = {}".format(
            round(
                recall_score(
                    y_true_pos, y_pred_pos, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    logger.info(
        "POS Precision = {}".format(
            round(
                precision_score(
                    y_true_pos, y_pred_pos, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    logger.info(
        "POS F1 = {}".format(
            round(
                f1_score(y_true_pos, y_pred_pos, average="weighted", zero_division=1), 3
            )
        )
    )
    logger.info("POS AUC = {}".format(round(micro_roc_auc_ovr, 3)))
    return


# Evaluate a NER run
@nlptimer
def evaluate_ner_run(output_file):
    label_names = [
        "DATE",
        "DURATION",
        "LOCATION",
        "MISC",
        "O",
        "MONEY",
        "NUMBER",
        "ORDINAL",
        "ORGANIZATION",
        "PERCENT",
        "PERSON",
        "SET",
        "TIME",
    ]

    gold_file = "test-local.tsv.gz"
    y_true_ner = []
    y_pred_ner = []

    with gzip.open(gold_file, "rt") as fh:
        for i, ln in enumerate(fh):
            if i == 0:
                if len(ln.strip().split("\t")) != 4:
                    print(
                        "[ERROR]: The file {} is not formatted correctly".format(
                            gold_file
                        )
                    )
                    logger.error(
                        "[ERROR]: The file {} is not formatted correctly".format(
                            gold_file
                        )
                    )
                    sys.exit(-1)
            ln = ln.strip()
            if ln:
                fld = ln.split("\t")
                y_true_ner.append(fld[3])

    ref_len = i

    for i, ln in enumerate(output_file.split("\n")):
        if i == 0:
            if len(ln.strip().split("\t")) != 3:
                print("[ERROR]: The data output_file is not formatted correctly.")
                data.error("[ERROR]: The data output_file is not formatted correctly.")
                sys.exit(-1)

        ln = ln.strip()
        if ln:
            fld = ln.split("\t")
            y_pred_ner.append(fld[2])

    pred_len = i

    if pred_len != ref_len:
        print(
            "[ERROR]: The reference file and prediction file are not the same length."
        )
        logger.error(
            "[ERROR]: The reference file and prediction file are not the same length."
        )
        sys.exit(-1)

    new_true_ner = []
    new_pred_ner = []
    # Remove the 'O' cases which are the majority class.
    for a, b in zip(y_true_ner, y_pred_ner):
        if a != "O" and b != "O":
            new_true_ner.append(a)
            new_pred_ner.append(b)
    y_true_ner = new_true_ner
    y_pred_ner = new_pred_ner

    print("[INFO] Classification report for NER predictions.")
    print(
        classification_report(
            y_true_ner, y_pred_ner, labels=label_names, zero_division=1
        )
    )
    logger.info("[INFO] Classification report for NER predictions.")
    logger.info(
        classification_report(
            y_true_ner, y_pred_ner, labels=label_names, zero_division=1
        )
    )

    df = confusion_matrix(y_true_ner, y_pred_ner, labels=label_names)
    print("INFO] Confusion Matrix for NER Predictions")
    print(df)
    logger.info("[INFOR] Confusion Matrix for NER Predictions")
    logger.info(df)

    label_binarizer = LabelBinarizer().fit(y_true_ner)
    y_onehot_test = label_binarizer.transform(y_pred_ner)
    x_onehot_true = label_binarizer.transform(y_true_ner)
    print("[INFO] NER Weighted Micro Scores")
    logger.info("[INFO] NER Weighted Micro Scores")
    micro_roc_auc_ovr = roc_auc_score(
        y_onehot_test, x_onehot_true, multi_class="ovr", average="micro"
    )
    print(
        "NER Recall = {}".format(
            round(
                recall_score(
                    y_true_ner, y_pred_ner, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    print(
        "NER Precision = {}".format(
            round(
                precision_score(
                    y_true_ner, y_pred_ner, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    print(
        "NER F1 = {}".format(
            round(
                f1_score(y_true_ner, y_pred_ner, average="weighted", zero_division=1), 3
            )
        )
    )
    print("NER AUC = {}".format(round(micro_roc_auc_ovr, 3)))
    logger.info(
        "NER Recall = {}".format(
            round(
                recall_score(
                    y_true_ner, y_pred_ner, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    logger.info(
        "NER Precision = {}".format(
            round(
                precision_score(
                    y_true_ner, y_pred_ner, average="weighted", zero_division=1
                ),
                3,
            )
        )
    )
    logger.info(
        "NER F1 = {}".format(
            round(
                f1_score(y_true_ner, y_pred_ner, average="weighted", zero_division=1), 3
            )
        )
    )
    logger.info("NER AUC = {}".format(round(micro_roc_auc_ovr, 3)))
    return


# Utility function to write a gzip conllu file.
# Example:
# output_text = ne_spacy_weblg('input.txt.gz')
# write_conllu_gzip('spacy-weblg.conllu.gz, output_text)
@nlptimer
def write_conllu_gzip(output_file, output_text):
    with gzip.open(output_file, "wt") as fh:
        fh.write(output_text)
    return


# Utility function to get number of lines for tqdm progress bar.
# Complete file line length should be 249212.
@nlptimer
def get_file_len(input_file):
    with gzip.open(input_file, "rt") as fh:
        num_lines = sum(1 for line in fh)
    return num_lines


# Utility function to dowonlaod a pretrained SpcCy model
@nlptimer
def get_spacy_model(model_name):
    spacy.cli.download(model_name)
    nlp_en = spacy.load(model_name)


def eprint(*args, **kwargs):
    print(*args, file=sys.stderr, **kwargs)

---


In [ ]:
# Monkeypatch Spacy tokenizer to a whitespace only tokenizer.
# You should NOT MODIFY this function. Do so at your own peril.


class SpWhitespaceTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab

    def __call__(self, text):
        words = text.split(" ")
        spaces = [True] * len(words)
        # Avoid zero-length tokens
        for i, word in enumerate(words):
            if word == "":
                words[i] = " "
                spaces[i] = False
        # Remove the final trailing space
        if words[-1] == " ":
            words = words[0:-1]
            spaces = spaces[0:-1]
        else:
            spaces[-1] = False
        return Doc(self.vocab, words=words, spaces=spaces)

### <font color=purple>END UTILITY FUNCTIONS</font>


---


#### Below is a template function that shows you how to walk the input file, which has one tokenized sentence per line, and output a tsv file with one parsed token per line. Each output sentence is separated by a single blank line. This code does not show you how to configure the parsing pipeline for each of the three libraries -- that is your task.


In [ ]:
def example_parser(input_file):
    output_file = []
    num_lines = get_file_len(input_file)
    with gzip.open(input_file, "rt") as fh:
        for ln in tqdm(fh, total=num_lines):
            tokens = ln.strip().split(" ")
            for i, t in enumerate(tokens):
                output_file.append("{}\t{}\t{}".format(i + 1, t, "A_LABEL"))
            output_file.append("")
    return "\n".join(output_file)


# Run on the sample file.
output_text = example_parser("test.txt.gz")
lines = output_text.split("\n")
# Now print the first 20 tokens from the tokenized output.
for i, ln in enumerate(lines):
    print(ln)
    if i == 25:
        break

---


### Data format and Requirements


---


In [ ]:
def dump_text(input_file):
    with gzip.open(input_file, "rt") as fh:
        for i, ln in enumerate(fh):
            print(ln.strip())
            if i == 25:
                break
    return


# Dump the text input of the test-local.txt file
print("--------------------------------------------")
print("Dump the first few lines of test-local.txt.gz")
print("--------------------------------------------")
dump_text("test-local.txt.gz")
print("--------------------------------------------")
print("Dump the first few lines of test-local.tsv.gz")
print("--------------------------------------------")
# Now dump a portion of the labelled file
dump_text("test-local.tsv.gz")

### I few simple observations. The input files are simple text, with one sentence per line. Note that the text is already tokenizedto remove issues with libbrary parsers. The output uses the CONL formatting convention, which has tab seperated output lines, one per token. There is always an empty space between the sentences. Note that the gold label data has both the POS and NER tag for each token in a single file. Your output should only be the task you are doing. That is, the gold label data has 4 tab separated values for each token but your output should onlt be three tab seperated columns: `position \t word \t tag`. For POS tagging, the tag would be a POS tag and for NER tagging, the tag would be an ner tag.


---


## <font color=purple>SpaCy POS Parser (4 Marks)</font>


---


In [ ]:
# You only need to run this function once. You can use this model for all of the experiments up to the model you train yourself.
get_spacy_model("en_core_web_sm")

---


In [ ]:
# YOU CODE GOES HERE
@nlptimer
def spacy_pos_tagger(input_file):
    output_text_lines = []
    return

In [ ]:
# Wrapper function to evaluate the output and write out the required files for POS tagging.

# Note that the function above is expected to return all of the lines including newlines in a
# single string. The only valid lines would be in CONLL format with a single space after each
# sentence.

print("\n [INFO] Running the SpaCy POS tagger for the model en_core_web_sm. \n")
logger.info("\n [INFO] Running the SpaCy POS tagger for the model en_core_web_sm. \n")
output_text = spacy_pos_tagger("test-local.txt.gz")
evaluate_pos_run(output_text)
write_conllu_gzip("spacy_sm_pos_test_local.tsv.gz", output_text)

print("[INFO] Generating POS tags for the holdout_set.")
output_text = spacy_pos_tagger("test.txt.gz")
write_conllu_gzip("spacy_sm_pos_test.tsv.gz", output_text)

---


## <font color=purple>SpaCy NER Parser (3 Marks)</font>


---


#### Now that you have the POS tagger working, modify your code to generate NER tags instead of POS tags. This should be relatively easy to do, with one caveat, the NER tagger will only tag words that it believes to be an entity. So you will need to test the tag generated for each token, and if there is no tag, you should tag the token using the tag 'O' such that every token you ouput has a tag. You can look at the ground truth labels for an example.


In [ ]:
# YOUR CODE GOES HERE
@nlptimer
def spacy_ner_tagger(input_file, model_name):
    output_text_lines = []
    return

In [ ]:
# Wrapper code to run your tagging function, and generate the required output for NER tagging.

# Same caveats apply to the above about the expected format of the string returned. The only difference
# here is that the model_name is passed into your functions so that you can reuse it for the model you
# train.

print("\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm. \n")
logger.info("\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm. \n")
output_text = spacy_ner_tagger("test-local.txt.gz", "en_core_web_sm")
write_conllu_gzip("spacy_sm_ner_test_local.tsv.gz", output_text)
evaluate_ner_run(output_text)

# Now try to filter the tags to see if the results get better.
print(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm with tag correction. \n"
)
logger.info(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm with tag correction. \n"
)
output_text = fix_default_ner_tags(output_text)
write_conllu_gzip("spacy_sm_ner_test_local_fixed.tsv.gz", output_text)
evaluate_ner_run(output_text)

# When you have the above function working, you should generate the output for the holdout data
print(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm for holdout set. \n"
)
logger.info(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm for holdout set. \n"
)
output_text = spacy_ner_tagger("test.txt.gz", "en_core_web_sm")
write_conllu_gzip("spacy_sm_ner_test.tsv.gz", output_text)

# When you have the above function working, you should generate the output for the holdout data with tag correction.
print(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm for holdout set. \n"
)
logger.info(
    "\n [INFO] Running the SpaCy NER tagger for the model en_core_web_sm for holdout set. \n"
)
ouput_text = fix_default_ner_tags(output_text)
write_conllu_gzip("spacy_sm_ner_test_fixed.tsv.gz", output_text)

---


## <font color=purple>Train a SpaCy NER Parser (5 Marks)</font>


---


##### You should be wondering at this point, why are the NER results so bad using SpaCy? The answer will hopefully be obvious to you if you look carefully at the gold labels versus the predicted labels -- They are quite different. This happens quiet often in Named Entity Recognition. Every collection defined the labels they think are important, and they tend to be far less consistent than the POS tagging. So using out-of-the-box moodels tends to be fine for most POS tasks (domain specific tasks excluded such as Law and Health where the vocabulary distriribution is not similar to the training data.)

##### Feel free to run any of the larger SpaCy models to convince yourself that this is the issue. In my experiments the results are only marginally better for the largest pretrained SpaCy model. So, we now want to train our own NER model. I have included the data you need to do this in the proper format. The two relevant files are `train.spacy` and `dev.spacy`. Using these two files, a little reading, and a lot of patience training the new model should greatly imporve the quality of the NER tagging model. There are two ways to accomplish this:

- The hard way by writing a training function; or
- The easy way using the harness built into SpaCy along with a configuration file you must create.

##### If you do it the easy way, you just need to create a valid config file and run the command line option commented out in the cell below. This will take many hours to finish unless you have a really good computer (News Flash ML is Hard), so be patient. Run it overnight. You should not have to run it to completion (but you can). Each epoch will create a foleder called `model-last` and the best one will be saved as `model-best`. After a about 5 epochs, you should observe that the model has converged, and each subsequent epoch will show little or no improvement. When it converges, you should be able to kill the trainer if you wish. Just make sure that the ``model-best'' model you have trained works correctly. Once you have trained your new model, write a function that will load your best model instead of the pre-trained model, and evaluate your local test file and generate the run for the holdout test file. If you did it correctly, you will see significantly better evaluation results for NER tagging.

##### It is about 4x to 5x slower to run the trainer in a Jupyter notebook than it is to run it from the command line. So if you want to run the training step from the command line, feel free. Just make sure that you redirect the output from the run to a log file and submit it.


In [ ]:
# YOUR CODE GOES HERE
# TODO Create the configuration file and run the command below, or write a training function to train your model.
# !python -m spacy train config.cfg --output ./ --paths.train ./train.spacy --paths.dev ./dev.spacy

In [ ]:
# Once you have a model-best folder containing your new model rerun the tagging and evaluation process by modifying
# the function below.
# These functions assume you have your trained model in the same directory as the jupyter notebook and that it is in a
# folder called "model-best'. If you have created your own training harness, modify the input as required.

print("\n Running my NER tagger. \n")
logger.info("\n Running my NER tagger. \n")
output_text = spacy_ner_tagger("test-local.txt.gz", "model-best")
write_conllu_gzip("spacy_best_ner_test_local.tsv.gz", output_text)
evaluate_ner_run(output_text)

# When you have the above function working, you should generate the output for the holdout data
output_text = spacy_ner_tagger("test.txt.gz", "model-best")
write_conllu_gzip("spacy_best_ner_test.tsv.gz", output_text)

---


## <font color=purple>Summary Results (3 Marks)</font>


---


#### Please complete the table below based on the results of your final experimental runs for the local-test.txt.gz file.


---


### <center>TABLE 1 - Evaluation Metrics</center><br>


| **Model**    | **F1 Score** | **Precision Score** | **Recall Score** | **OvA AUC** |
| ------------ | ------------ | ------------------- | ---------------- | ----------- |
| `spaCy POS`  |              |                     |                  |             |
| `spaCy NER`  |              |                     |                  |             |
| `custom NER` |              |                     |                  |             |


---


##### That's it for the assignmnet. When you have ran everything and generated all of the required output, rename your working directory from A1-v1.1 to your "student email name", i.e. s583762, and create a zip file including the student id directory name. So if you unzipped the file you just created, you should see something like this where sXXXXXX is your student id as in your email address.

```
# Log files, config file if you used one, and the Readme.md you were given, modified with any notes relevant to your solution.
sXXXXX/COMP4703A1.log
sXXXXX/COMP4703A1.log.bak
sXXXXX/config.cfg
sXXXXX/README.md

# Checkpoints and full run of your project code.
sXXXXX/A1.ipynb
sXXXXX/.ipynb_checkpoints/A1-checkpoint.ipynb

# The last epoch trained.
sXXXXX/model-last
sXXXXX/model-last/ner
sXXXXX/model-last/ner/moves
sXXXXX/model-last/ner/cfg
sXXXXX/model-last/ner/model
sXXXXX/model-last/tokenizer
sXXXXX/model-last/vocab
sXXXXX/model-last/vocab/vectors
sXXXXX/model-last/vocab/lookups.bin
sXXXXX/model-last/vocab/strings.json
sXXXXX/model-last/vocab/key2row
sXXXXX/model-last/vocab/vectors.cfg
sXXXXX/model-last/config.cfg
sXXXXX/model-last/meta.json

# The "best" model trained
sXXXXX/model-best
sXXXXX/model-best/ner
sXXXXX/model-best/ner/moves
sXXXXX/model-best/ner/cfg
sXXXXX/model-best/ner/model
sXXXXX/model-best/tokenizer
sXXXXX/model-best/vocab
sXXXXX/model-best/vocab/vectors
sXXXXX/model-best/vocab/lookups.bin
sXXXXX/model-best/vocab/strings.json
sXXXXX/model-best/vocab/key2row
sXXXXX/model-best/vocab/vectors.cfg
sXXXXX/model-best/config.cfg
sXXXXX/model-best/meta.json

# Original data files. You can remove them from your submission as we will overwrite them anyway.
sXXXXX/train.spacy
sXXXXX/dev.spacy
sXXXXX/test-local.txt.gz
sXXXXX/test-local.tsv.gz
sXXXXX/test.txt.gz

# POS Run files. There should be 2 of these, one from test-local and one for the holdhout test set.
sXXXXX/spacy_sm_pos_test_local.tsv.gz
sXXXXX/spacy_sm_pos_test.tsv.gz

# NER Run files. There should be 6 of these, one from test-local and one for the holdhout test set for
# the unchanged tags with vanilla model, the remaped tags of the vanilla model, and the custom model you
# trained.
sXXXXX/spacy_best_ner_test_local.tsv.gz
sXXXXX/spacy_best_ner_test.tsv.gz
sXXXXX/spacy_sm_ner_test_fixed.tsv.gz
sXXXXX/spacy_sm_ner_test_local_fixed.tsv.gz
sXXXXX/spacy_sm_ner_test_local.tsv.gz
sXXXXX/spacy_sm_ner_test.tsv.gz


# Optional file if you train your model from the command line for performance reasons. If you wrote custom
# trainer code or ran everything in the jupyter notebook, you can ignore this requirement.
sXXXXX/my_trainer.log

```

##### As you can see, there are quite a few files you need to submit. We need these files to validate your solutions. We are less interested in the files for "logcal-test" than for the real holdout set "test" so these are essential ans they rwill be evaluateed using the habels that you did not receive. This is common proactice in NLP leaderboard datasets, to prevent anyone from overtraining on the hold out set. Achieving results consistent with our reference implementation (or beating it), will get you the most marks. See the rubric for an exact breakdown of how marks will be allocated.
